# E02b — blend the main model with XGBoost (validation only)

**Runtime:** a CPU runtime is enough (no GPU needed). Runtime → Run all; about 3–5 minutes.

**Rule fixed before scoring (experiment ledger, E02b):** for each seed s, `p = 0.5 × main model (E01) + 0.5 × XGBoost`,
predicted class = most likely class. Weight 0.5 is not tuned. Deep input = E01, `main_model_v1_news_p41_preds.csv.gz`, already in Drive.

Steps: (1) rerun XGBoost unchanged, now saving its probabilities; (2) blend. Outputs in `MyDrive/Senior/results/`:
`xgb_e02b.json`, `xgb_e02b_preds.csv.gz`, `blend_e02b.json`. Test block stays sealed.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SENIOR = '/content/drive/MyDrive/Senior'
PRICE_DIR, OUT_DIR = f'{SENIOR}/prices', f'{SENIOR}/results'

In [ ]:
!rm -rf /content/Senior-Project && git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1
import hashlib, os, xgboost
print('xgboost', xgboost.__version__)
for f in ['scripts/baselines_tabular.py', 'scripts/blend_preds.py']:
    assert os.path.exists(f), f'stop: push {f} from your PC, then run again'
    print(f, hashlib.sha256(open(f, 'rb').read()).hexdigest()[:12])
assert 'predict_proba' in open('scripts/baselines_tabular.py', encoding='utf-8').read(), 'stop: push the latest baselines_tabular.py (saves probabilities)'
assert os.path.exists(f'{OUT_DIR}/main_model_v1_news_p41_preds.csv.gz'), 'stop: E01 predictions not found in Drive results'


In [ ]:
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo "$OUT_DIR"
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/
# Files must match the PC copies (hashes recorded 2026-10-15; sentiment file as stored in git, LF line endings)
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': '7369fd981c4ccffc017d6d5de50ed95c6a4ac4d843bbc6053366029ce614bb1b'}
for path, h in expected.items():
    got = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if got == h else 'MISMATCH ', path)
    assert got == h, 'stop: a file differs from the PC copy'

In [ ]:
!python tests/test_pipeline.py

In [ ]:
# 1) XGBoost, same fixed settings as the M2 baseline, now also writing every probability
!python scripts/baselines_tabular.py --out "$OUT_DIR/xgb_e02b.json"


In [ ]:
# 2) Blend with E01 (w = 0.5, seed s with seed s); prints each input alone, the blend, and the gap to the pass bar
!python scripts/blend_preds.py --deep "$OUT_DIR/main_model_v1_news_p41_preds.csv.gz" --deep-model multiscale_sector \
    --xgb "$OUT_DIR/xgb_e02b_preds.csv.gz" --weight 0.5 --out "$OUT_DIR/blend_e02b.json"
